# Extended Case 2 — Oxford Cereal

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: case-import
source("lessons/_includes/course-tools.R")
oxford <- read.csv("data/business/extended-oxford-summary.csv")
knitr::kable(oxford)
n <- oxford$n[1]
xbar <- oxford$sample_mean_g[1]
sigma <- oxford$population_sd_g[1]
mu0 <- oxford$target_mean_g[1]
alpha <- oxford$alpha[1]
stopifnot(n == 25, sigma == 15, mu0 == 368, xbar == 372.5)


- `data/business/extended-oxford-summary.csv`; guide: `data/business/README.md`.
- `sigma` is assumed known in the textbook example; it is not estimated from these 25 boxes.
- A Z procedure requires the stated known SD and an appropriate sampling/normal model.

## 3. Separate variation between boxes and between sample means


In [ ]:
se <- sigma / sqrt(n)
knitr::kable(data.frame(sample_size = n, sample_mean_g = xbar,
  difference_from_target_g = xbar - mu0, process_sd_g = sigma,
  standard_error_g = se), digits = 2)


- `sigma = 15` g describes variation between individual boxes under the model.
- `se = 3` g describes variation between means of independent samples of 25 boxes.
- Increasing sample size reduces the standard error; it does not mechanically reduce process variation.

## 4. See the sampling distribution

- This simulation generates **hypothetical** normal-process weights under a 368 g mean and 15 g SD.
- Its values are not observed Oxford measurements.


In [ ]:
set.seed(42001)
sim_means <- replicate(10000, mean(rnorm(n, mean = mu0, sd = sigma)))
ggplot(data.frame(sample_mean = sim_means), aes(sample_mean)) +
  geom_histogram(bins = 35, fill = petra_accent, colour = "white") +
  geom_vline(xintercept = xbar, colour = petra_ink, linewidth = 1) +
  labs(x = "Mean of 25 simulated boxes (g)", y = "Simulated samples")


- `set.seed()` makes the simulation reproducible.
- `replicate()` repeats the hypothetical sampling experiment.
- The vertical line marks the reported sample mean, 372.5 g.

## 5. Test the process mean

- **Null:** the population mean is 368 g.
- **Alternative:** the population mean differs from 368 g.
- **Decision threshold:** 5%, two-sided.


In [ ]:
z <- (xbar - mu0) / se
p_value <- 2 * pnorm(-abs(z))
ci <- xbar + c(-1, 1) * qnorm(1 - alpha / 2) * se
knitr::kable(data.frame(z = z, p_value = p_value,
  lower_mean_g = ci[1], upper_mean_g = ci[2], reject = p_value < alpha), digits = 4)


- `pnorm()` gives the normal tail probability; the two tails account for deviations in either direction.
- The result is **Z = 1.50**, with p about **0.134**.
- The 95% interval for the process mean is approximately **366.62–378.38 g**.
- The evidence is insufficient to reject the 368 g mean at 5% under the stated assumptions.
- This does not prove exact centering, individual-box compliance or a low underfill rate.

## 6. Independent packaging review

1.  Explain why `sd()` or `t.test()` on the single summary row cannot analyse 25 individual boxes.
2.  Recalculate the test and interval for a **hypothetical** sample mean of 375 g, holding the other inputs fixed.
3.  Compare the standard errors for hypothetical independent sample sizes 25 and 100 with the same known process SD.
4.  Explain why a narrow interval for the mean does not guarantee that individual boxes meet a weight requirement.
5.  Describe Type I and Type II errors as unnecessary process intervention and missed process changes.
6.  Specify the raw measurements and sampling records needed to investigate underfilling and process stability.


**Manual setup — run once in the R Console if needed.** This cell is excluded from Run All.

```r
#| eval: false
# Independent task: change the summary inputs and label the scenario hypothetical.
# Use (sample_mean - target) / (known_sd / sqrt(sample_size)).
```


## Submission

- One comparison table for the reported and hypothetical scenarios.
- A sampling-distribution chart with clearly labelled simulated values.
- Five briefing bullets covering evidence, decision, assumptions, limits and further data.
